In [1]:
import string
import pandas as pd
from datetime import datetime, timedelta, timezone
from icalendar import Calendar, Event
import hashlib
import pytz

In [2]:
time_zone = "America/Toronto"

data_path = '2026_35_summer'
league_name = 'Summer League 2026 — 35+'
location = 'Bill Bolton Arena'
team = 'Lakers'
event_duration = 60 # In minutes

home_desc = 'home'
away_desc = 'away'

In [3]:
tz = pytz.timezone("America/Toronto")

In [4]:
def string_to_uid(s):
    return hashlib.sha1(s.encode("utf-8")).hexdigest() + "@hockey"

def make_uid(row, team=team):
    raw = f"BBA-{row.Date}-{row.Time}-{team}"
    return string_to_uid(raw)

In [5]:
def longest_common_prefix(strings):
    strings = list(strings)
    if not strings:
        return ""

    # sort so extremes determine prefix
    strings.sort()

    first = strings[0]
    last = strings[-1]

    i = 0
    while i < len(first) and i < len(last) and first[i] == last[i]:
        i += 1

    return first[:i]

In [6]:
schedule_df = pd.read_csv(data_path+'.csv')
schedule_df.head()

,Date,Game,Time,Home,Visitor
0,2026-06-02,1,6:30pm,Cowboys,Lakers
1,2026-06-02,1,7:30pm,Hurricanes,Titans
2,2026-06-02,1,8:30pm,Celtics,Bengals
3,2026-06-02,1,9:30pm,Steelers,Seahawks
4,2026-06-09,2,6:30pm,Seahawks,Lakers


In [7]:
team_df = schedule_df[
    (schedule_df['Home'].str.contains(team))
    |
    (schedule_df['Visitor'].str.contains(team))
]
display(team_df)

,Date,Game,Time,Home,Visitor
0,2026-06-02,1,6:30pm,Cowboys,Lakers
4,2026-06-09,2,6:30pm,Seahawks,Lakers
10,2026-06-16,3,8:30pm,Lakers,Hurricanes
15,2026-06-23,4,9:30pm,Bengals,Lakers
17,2026-07-07,5,7:30pm,Steelers,Lakers
20,2026-07-14,6,6:30pm,Lakers,Titans
27,2026-07-21,7,9:30pm,Celtics,Lakers
28,2026-07-28,8,6:30pm,Cowboys,Lakers
33,2026-08-04,9,7:30pm,Seahawks,Lakers


In [8]:
playoff_df = schedule_df[~schedule_df['Date'].isin(team_df['Date'])]
playoff_df

,Date,Game,Time,Home,Visitor
36,2026-08-11,Playoff Round 1 (A),6:30pm,3rd,6th
37,2026-08-11,Playoff Round 1 (B),7:30pm,1st,8th
38,2026-08-11,Playoff Round 1 (C),8:30pm,2nd,7th
39,2026-08-11,Playoff Round 1 (D),9:30pm,4th,5th
40,2026-08-18,Playoff Round 2 (1),6:30pm,Loser B,Loser D
41,2026-08-18,Playoff Round 2 (2),7:30pm,Winner B,Winner D
42,2026-08-18,Playoff Round 2 (3),8:30pm,Winner C,Winner A
43,2026-08-18,Playoff Round 2 (4),9:30pm,Loser C,Loser A
44,2026-08-25,Finals (Champ),6:30pm,Winner 2,Winner 3
45,2026-08-25,Finals (3rd),7:30pm,Loser 2,Loser 3


In [9]:
cal = Calendar()
cal.add("x-wr-calname", f"{league_name} Team {team} Schedule")
cal.add("x-wr-timezone", time_zone)

In [10]:
for row in team_df.itertuples(index=False):
    start_time = datetime.strptime(
        f"{row.Date} {row.Time}",
        "%Y-%m-%d %I:%M%p"
    )
    start_time = tz.localize(start_time)
    end_time = start_time + timedelta(minutes=event_duration)

    at_home = row.Home == team
    opposite_team = row.Visitor if at_home else row.Home
    description = f"Game {row.Game} against {opposite_team} ({home_desc if at_home else away_desc})"

    event = Event()
    event.add("uid", make_uid(row, team))
    event.add("dtstamp", datetime.now(timezone.utc))
    event.add("summary", f"{league_name}")
    event.add("description", description)
    event.add("dtstart", start_time)
    event.add("dtend", end_time)
    event.add("location", location)

    cal.add_component(event)
    

In [11]:
for date in playoff_df['Date'].unique():
    start_time = min(playoff_df[playoff_df['Date']==date]['Time'])
    start_time = datetime.strptime(
        f"{date} {start_time}",
        "%Y-%m-%d %I:%M%p"
    )
    start_time = tz.localize(start_time)

    num_games = sum(playoff_df['Date']==date)
    end_time = start_time + timedelta(minutes = num_games * event_duration)

    event_name = longest_common_prefix(playoff_df[playoff_df['Date']==date]['Game'])
    while event_name and event_name[-1]==' ':
        event_name = event_name[:-1]
    if not event_name:
        event_name = "Playoff Game"
    
    event = Event()
    event.add("uid", string_to_uid(f"BBA-{date}-{event_name}-{team}"))
    event.add("dtstamp", datetime.now(timezone.utc))
    event.add("summary", f"{league_name}")
    event.add("description", event_name)
    event.add("dtstart", start_time)
    event.add("dtend", end_time)
    event.add("location", location)

    cal.add_component(event)

In [12]:
with open(f'{data_path}.ics', 'wb') as f:
    f.write(cal.to_ical())